In [0]:
from pyspark.sql import functions as F

In [0]:
SOURCE_CATALOG_NAME = 'beverage_sales'
SOURCE_SCHEMA_NAME = 'silver'
SOURCE_TABLE_NAME = 'sales'

TARGET_CATALOG_NAME = 'beverage_sales'
TARGET_SCHEMA_NAME = 'gold'
TARGET_TABLE_NAME = 'dim_brand'

UNKNOWN_KEY = -1

In [0]:
df_unknown_member = spark.createDataFrame(
    [(UNKNOWN_KEY, 'UNKNOWN', 'UNKNOWN')],
    'brand_key bigint, brand_code string, brand_name string'
)

In [0]:
df_source = spark.table(f'{SOURCE_CATALOG_NAME}.{SOURCE_SCHEMA_NAME}.{SOURCE_TABLE_NAME}')

## Cardinality check

`dropDuplicates` on a natural key is only safe when that key functionally
determines the remaining attributes. A future file where one `brand_code` carries two different
attribute sets fails here instead of silently keeping an arbitrary row.

In [0]:
df_invalid_mapping = (
    df_source
    .groupBy('brand_code')
    .agg(F.countDistinct('brand_name').alias('attribute_count'))
    .filter(F.col('attribute_count') > 1)
)

assert df_invalid_mapping.count() == 0, 'brand_code does not uniquely determine brand_name'

In [0]:
df_dim_brand = (
    df_source
    .select('brand_code', 'brand_name')
    .dropDuplicates(['brand_code'])
    .withColumn('brand_key', F.abs(F.xxhash64(F.col('brand_code'))))
    .select(
        'brand_key',
        'brand_code',
        'brand_name'
    )
    .unionByName(df_unknown_member)
)

In [0]:
df_dim_brand\
    .write\
    .mode('overwrite')\
    .saveAsTable(f'{TARGET_CATALOG_NAME}.{TARGET_SCHEMA_NAME}.{TARGET_TABLE_NAME}')